# 🔄 Actualizar Dashboard Clientes
Actualiza las **5 secciones** del dashboard de clientes.

| Sección | Fuente |
|---|---|
| Precios RPM | `IPC TODESCA.xlsx` · `Cuadro Mensual - Capítulos Nuevo.xlsx` |
| Actividad EsAE | `Base EsAE.xlsx` |
| Reservas | `pasivos reservas.xlsx` · `Reservas brutas y depósitos.xlsx` |
| RIGI | Datos hardcodeados — no requiere actualización |
| Internacional · Consensus | PDF LatinFocus → `python parse_latinfocus.py "archivo.pdf"` |

> **Nota:** el dashboard clientes lee los mismos archivos `.js` que el dashboard maestro. Correr este notebook es suficiente para refrescar las secciones relevantes.


In [1]:
import sys, os

DASHBOARD_DIR = r"C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard"
sys.path.insert(0, DASHBOARD_DIR)

import refresh as R

print(f"Dashboard: {DASHBOARD_DIR}")
print(f"Excel base: {R.BASE_EXCEL}")
print(f"Data dir:   {R.DATA_DIR}")
print()
print("Secciones a actualizar: Precios RPM · Actividad EsAE · Reservas")

Dashboard: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard
Excel base: C:\Users\fscalise\OneDrive - ECOGO S.A\BD
Data dir:   C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard\assets\data

Secciones a actualizar: Precios RPM · Actividad EsAE · Reservas


## 1 · Precios RPM

In [2]:
status = R.Status()
d = R.extract_precios(status)
if d:
    sz = R.save_data('precios', d)
    # Mostrar solo los datos RPM
    rpm = d.get('proyeccion', {})
    chart22 = d.get('chart22', [])
    print(f"✅ precios.js — {sz:,} bytes")
    if rpm and rpm.get('filas'):
        print(f"   Proyección RPM: {len(rpm['filas'])} capítulos · {rpm.get('header_periodo','')}")
    if chart22:
        ult = [p for p in chart22 if p.get('rpm_mm') is not None]
        if ult:
            u = ult[-1]
            rpm_pct = round(u['rpm_mm']*100, 2) if u['rpm_mm'] else '—'
            print(f"   Último RPM m/m: {rpm_pct}% ({u['fecha'][:7]})")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [OK]   Precios - Proyección RPM — hoja 2606, 10 filas
✅ precios.js — 62,837 bytes
   Proyección RPM: 10 capítulos · Proyección — RPM Eco Go · 2606
   Último RPM m/m: 2.52% (2026-04)
  [OK] Precios - Proyección RPM: hoja 2606, 10 filas


## 2 · Actividad EsAE

In [3]:
status = R.Status()
d = R.extract_emae_series(status)
if d:
    sz = R.save_emae_series(d)
    orig = d.get('original', [])
    print(f"✅ emae_series.js — {sz:,} bytes")
    if orig:
        print(f"   Serie original: {len(orig)} observaciones · último: {orig[-1].get('date','')}")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [---]  EMAE Series — no se encontró hoja de serie original (buscando: EMAE/Original/Serie/Mensual)
  [---]  EMAE Series — no se encontró hoja de sectores CE
  [---]  EMAE Series — no se encontró hoja de sectores SE
  [FAIL] EMAE Series — no se extrajo ningún dato de Base EsAE.xlsx. Verificar nombres de hojas: ['Hoja1']
  [WARN] EMAE Series: no se encontró hoja de serie original (buscando: EMAE/Original/Serie/Mensual)
  [WARN] EMAE Series: no se encontró hoja de sectores CE
  [WARN] EMAE Series: no se encontró hoja de sectores SE
  [FAIL] EMAE Series: no se extrajo ningún dato de Base EsAE.xlsx. Verificar nombres de hojas: ['Hoja1']


## 3 · Reservas

In [4]:
status = R.Status()
d = R.extract_reservas(status)
if d:
    sz = R.save_data('reservas', d)
    rin = d.get('rin', {})
    g5  = d.get('g5', [])
    print(f"✅ reservas.js — {sz:,} bytes")
    if rin:
        print(f"   RIN: {len(rin.get('dates',[]))} fechas · {len(rin.get('rows',[]))} filas")
    if g5:
        print(f"   G5 (reservas diarias): {len(g5)} observaciones · último: {g5[-1].get('d','')}")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [OK]   Reservas - RIN — 26 fechas - 21 filas
  [OK]   Reservas - G5 — 5500 dias - ultimo: 2025-06-04
✅ reservas.js — 376,440 bytes
   RIN: 26 fechas · 21 filas
   G5 (reservas diarias): 5500 observaciones · último: 2025-06-04
  [OK] Reservas - RIN: 26 fechas - 21 filas
  [OK] Reservas - G5: 5500 dias - ultimo: 2025-06-04


## 4 · RIGI
> Los datos del RIGI están hardcodeados en la página. **No requiere actualización desde Excel.**

In [5]:
print("ℹ️  RIGI: datos embebidos en rigi-clientes.html — sin acción requerida.")

ℹ️  RIGI: datos embebidos en rigi-clientes.html — sin acción requerida.


## 5 · Internacional · Consensus (LatinFocus)
> Requiere tener el PDF LatinFocus del mes correspondiente. Correr manualmente desde terminal.


In [6]:
import subprocess, os

# Ajustar al nombre del PDF del mes actual
PDF_NAME = 'LatinFocus Consensus Forecast - May 2026.pdf'
PDF_PATH = os.path.join(DASHBOARD_DIR, PDF_NAME)

if not os.path.exists(PDF_PATH):
    print(f'⚠️  PDF no encontrado: {PDF_PATH}')
    print('   Copiá el PDF a la carpeta del dashboard y actualizá PDF_NAME.')
else:
    result = subprocess.run(
        ['python', os.path.join(DASHBOARD_DIR, 'parse_latinfocus.py'), PDF_PATH],
        capture_output=True, text=True, cwd=DASHBOARD_DIR
    )
    print(result.stdout)
    if result.returncode != 0:
        print('❌ Error:', result.stderr)


⚠️  PDF no encontrado: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard\LatinFocus Consensus Forecast - May 2026.pdf
   Copiá el PDF a la carpeta del dashboard y actualizá PDF_NAME.


## ✅ Resumen

In [7]:
from datetime import datetime

archivos = ['precios.js', 'emae_series.js', 'reservas.js', 'internacional2.js']
print(f"{'Archivo':<22} {'Tamaño':>12} {'Modificado'}")
print('-' * 55)
for f in archivos:
    p = os.path.join(R.DATA_DIR, f)
    if os.path.exists(p):
        sz = os.path.getsize(p)
        mt = datetime.fromtimestamp(os.path.getmtime(p)).strftime('%d/%m %H:%M')
        print(f"{f:<22} {sz:>10,} b  {mt}")
    else:
        print(f"{f:<22} {'—':>12}")
print()
print('Listo. Recargá el dashboard clientes en el navegador.')


Archivo                      Tamaño Modificado
-------------------------------------------------------
precios.js                 62,837 b  10/06 14:20
emae_series.js            108,319 b  27/05 15:15
reservas.js               376,440 b  10/06 14:20
internacional2.js           7,769 b  10/06 13:07

Listo. Recargá el dashboard clientes en el navegador.
